# Stage 18 — data / lexicon / tracking AUDIT (CPU, do first)
Answers before any retrain/GPU: (1) is there multi-view data to add, and does the
cache use every clip? (2) are lexical words in a fixed ~6000 vocab (OOV rate ->
is item-2 lexicon decoding viable & leakage-free)? (3) which signers track badly?
**Attach BOTH** `gaurs86/wita-full-english-122signers` (raw) **and**
`gaurs86/wita-full-english-landmark-cache`. No GPU.


## Cell 1 — clone + deps


In [ ]:
import sys, subprocess as sp
sp.run('rm -rf /kaggle/working/wita_v2', shell=True)
sp.run("git clone -b stage13b-paper-replication "
       "'https://github.com/Gaurs86/WiTA-v2.git' '/kaggle/working/wita_v2'", shell=True, check=True)
sys.path.insert(0, '/kaggle/working/wita_v2')
for _m in [m for m in sys.modules if m.split('.')[0] in ('stage16','stage17','stage18','datasets','models')]:
    del sys.modules[_m]
sp.run('pip -q install wordfreq', shell=True)
print('ready')


## Cell 2 — item 1: data structure + multi-view detection


In [ ]:
from stage18.audit import audit_structure, audit_cache_vs_raw
g = audit_structure()      # prints clips/dirs/signers + max dirs per signer + suffixes
audit_cache_vs_raw()       # raw clips vs cached npz (are we using all the data?)


## Cell 3 — item 2: lexical OOV vs a fixed 6000-word vocab


In [ ]:
from stage18.audit import audit_lexicon
lex, union = audit_lexicon(top_k=6000)


## Cell 4 — item 4: per-signer tracking quality


In [ ]:
from stage18.audit import audit_tracking
rows = audit_tracking(out_json='/kaggle/working/tracking_stats.json')


## Read it
- **Cell 2:** if `MULTI-VIEW present = False` and cache coverage ~100%, item 1 is
  moot — we already use all the data; skip the re-extraction.
- **Cell 3:** if **test lex OOV vs top-6000 is ~0%**, item 2 (trie-beam over the
  6000 vocab) is the real headline lever and leakage-free. (Contrast: Stage 17b's
  train-lexicon covered only ~49%.)
- **Cell 4:** worst-tracking signers — we'll correlate with per-signer CER next to
  decide item 3 (augmentation) vs item 4 (re-track/smooth).
